# 🎭 Emotion Detection in Social Media
## Classifying Reddit Comments Using Machine Learning

**Course:** IE 423 — Machine Learning Applications in Industrial Engineering  
**Team:** DaSpace Mockers  
**Members:** Basil Sadlah, Saleh Yaish, Parsa Badiee, Mohammed Al-Hamami  
**Dataset:** GoEmotions (Google Research, 2020)  
**Emotion Framework:** Parrot's Emotion Model (6 primary emotions + neutral)

---

This notebook covers the full pipeline:
- Data loading and preprocessing
- Exploratory Data Analysis (EDA)
- Baseline model training (LR, SVM, kNN)
- Imbalance handling experiments (SMOTE, LSA, Class Weights)
- Hyperparameter tuning (GridSearch)
- Metaheuristic optimization (Genetic Algorithm)

# 01_load_data.py:

In [ ]:
import pandas as pd
from datasets import load_dataset
import os

# ── Create folder structure ──────────────────────────────────────────────────
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('visuals', exist_ok=True)

# ── Load GoEmotions (raw) ────────────────────────────────────────────────────
print("Loading GoEmotions dataset...")
dataset = load_dataset("google-research-datasets/go_emotions", "raw")
df = pd.DataFrame(dataset['train'])
print(f"Raw dataset shape: {df.shape}")
print(f"Unique comments: {df['id'].nunique()}")

# ── Aggregate by comment ID (one row per comment) ────────────────────────────
emotion_cols = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]

df_agg = df.groupby('id').agg(
    text=('text', 'first'),
    **{col: (col, 'max') for col in emotion_cols}
).reset_index()

print(f"Aggregated dataset shape: {df_agg.shape}")

# ── Apply Parrot mapping ─────────────────────────────────────────────────────
parrot_mapping = {
    'joy':      ['joy', 'amusement', 'excitement', 'optimism', 'pride', 'relief', 'gratitude', 'approval'],
    'love':     ['love', 'admiration', 'caring', 'desire'],
    'surprise': ['surprise', 'realization', 'confusion', 'curiosity'],
    'anger':    ['anger', 'annoyance', 'disapproval', 'disgust', 'embarrassment'],
    'sadness':  ['sadness', 'disappointment', 'grief', 'remorse'],
    'fear':     ['fear', 'nervousness'],
    'neutral':  ['neutral']
}

def assign_parrot_label(row):
    for parrot_class, emotions in parrot_mapping.items():
        if any(row[e] == 1 for e in emotions):
            return parrot_class
    return None

df_agg['parrot_label'] = df_agg.apply(assign_parrot_label, axis=1)

# ── Drop unlabeled rows ──────────────────────────────────────────────────────
df_agg = df_agg[df_agg['parrot_label'].notna()].reset_index(drop=True)
print(f"After dropping unlabeled: {len(df_agg)} rows")

# ── Save raw aggregated ──────────────────────────────────────────────────────
df_agg[['text', 'parrot_label']].to_csv('data/raw/go_emotions_raw.csv', index=False)
print("Saved to data/raw/go_emotions_raw.csv")

# ── Print class distribution ─────────────────────────────────────────────────
print("\nClass distribution:")
print(df_agg['parrot_label'].value_counts())

# 02_preprocess_data.py:

In [ ]:
import pandas as pd
import re
import os

# ── Load raw data ────────────────────────────────────────────────────────────
df = pd.read_csv('data/raw/go_emotions_raw.csv')
print(f"Loaded: {df.shape}")

# ── Text cleaning ────────────────────────────────────────────────────────────
def clean_text(text):
    text = str(text)
    text = text.encode('ascii', 'ignore').decode('ascii')  # remove encoding artifacts
    text = re.sub(r'http\S+', '', text)                    # remove URLs
    text = re.sub(r'\s+', ' ', text).strip()               # normalize whitespace
    text = text.lower()                                     # lowercase
    return text

df['text'] = df['text'].apply(clean_text)

# ── Remove empty rows after cleaning ────────────────────────────────────────
before = len(df)
df = df[df['text'].str.strip() != ''].reset_index(drop=True)
print(f"Removed {before - len(df)} empty rows after cleaning")

# ── Add features ─────────────────────────────────────────────────────────────
df['comment_length'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

# ── Save processed data ──────────────────────────────────────────────────────
df.to_csv('data/processed/combined_dataset.csv', index=False)
print(f"Saved processed dataset: {df.shape}")

# ── Print summary ─────────────────────────────────────────────────────────────
print("\nClass distribution:")
print(df['parrot_label'].value_counts())
print(f"\nAverage comment length: {df['comment_length'].mean():.1f} characters")
print(f"Average word count: {df['word_count'].mean():.1f} words")

# 03_basic_eda.py:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs('visuals', exist_ok=True)

df = pd.read_csv('data/processed/combined_dataset.csv')

# ── 1. Class distribution bar chart ─────────────────────────────────────────
plt.figure(figsize=(10, 5))
df['parrot_label'].value_counts().plot(kind='bar', color='steelblue')
plt.title('Emotion Class Distribution (Parrot Mapping)')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('visuals/01_class_distribution.png')
plt.show()

# ── 2. Comment length KDE by label ─────────────────────────────────────────────
plt.figure(figsize=(10, 5))
for label in df['parrot_label'].unique():
    subset = df[df['parrot_label'] == label]
    subset['comment_length'].plot(kind='kde', label=label)
plt.title('Comment Length Distribution by Emotion')
plt.xlabel('Comment Length (characters)')
plt.legend()
plt.tight_layout()
plt.savefig('visuals/02_comment_length_kde.png')
plt.show()

# ── 3. Word count boxplot by label ───────────────────────────────────────────
plt.figure(figsize=(10, 5))
df.boxplot(column='word_count', by='parrot_label')
plt.title('Word Count by Emotion')
plt.suptitle('')
plt.xlabel('Emotion')
plt.ylabel('Word Count')
plt.tight_layout()
plt.savefig('visuals/03_word_count_boxplot.png')
plt.show()

# ── 4. Summary table ─────────────────────────────────────────────────────────
summary = df.groupby('parrot_label').agg(
    count=('text', 'count'),
    avg_length=('comment_length', 'mean'),
    avg_words=('word_count', 'mean')
).round(2)

summary.to_csv('visuals/01_label_summary.csv')
print(summary)

# 04_model.py

## Cell 1 — **imports and load data**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
import os

os.makedirs('visuals', exist_ok=True)

# ── Load data ────────────────────────────────────────────────────────────────
df = pd.read_csv('data/processed/combined_dataset.csv')
X = df['text']
y = df['parrot_label']

# ── Train/test split (stratified) ───────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

## Cell 2 — **train/test split + TF-IDF**

In [ ]:
# ── Train/test split (stratified) ───────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

# ── TF-IDF vectorization ─────────────────────────────────────────────────────
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## Cell 3 — **helper function (plot_confusion_matrix)**

In [ ]:
# ── Helper: plot confusion matrix ────────────────────────────────────────────
def plot_confusion_matrix(y_test, y_pred, title, filename):
    labels = sorted(y_test.unique())
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'visuals/{filename}')
    plt.show()

## Cell 4 — **LR baseline**

In [ ]:
# Logistic Regression
lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)
print("\n=== Logistic Regression (Baseline) ===")
print(classification_report(y_test, y_pred_lr))
plot_confusion_matrix(y_test, y_pred_lr, 'Confusion Matrix — Logistic Regression', '04_cm_lr_baseline.png')

## Cell 5 — **SVM baseline**


In [ ]:
# SVM
svm = LinearSVC(class_weight='balanced', random_state=42, max_iter=2000)
svm.fit(X_train_tfidf, y_train)
y_pred_svm = svm.predict(X_test_tfidf)
print("\n=== SVM (Baseline) ===")
print(classification_report(y_test, y_pred_svm))
plot_confusion_matrix(y_test, y_pred_svm, 'Confusion Matrix — SVM', '05_cm_svm_baseline.png')

## Cell 6 — **kNN baseline**

In [ ]:
# kNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_tfidf, y_train)
y_pred_knn = knn.predict(X_test_tfidf)
print("\n=== kNN (Baseline) ===")
print(classification_report(y_test, y_pred_knn))
plot_confusion_matrix(y_test, y_pred_knn, 'Confusion Matrix — kNN', '06_cm_knn_baseline.png')

## Cell 7 — **GridSearch best model**

In [ ]:
# ── Best model: LR + GridSearch + Aggressive Class Weights ───────────────────
custom_weights = {
    'joy': 1, 'surprise': 2, 'anger': 2, 'love': 2,
    'neutral': 3, 'sadness': 5, 'fear': 20
}

param_grid = {
    'C': [0.1, 0.5, 1, 5, 10],
    'class_weight': ['balanced', custom_weights]
}

grid = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000),
    param_grid, cv=5, scoring='f1_macro', n_jobs=-1
)
grid.fit(X_train_tfidf, y_train)
print(f"\nBest params: {grid.best_params_}")
print(f"Best CV score: {grid.best_score_:.4f}")

y_pred_best = grid.predict(X_test_tfidf)
print("\n=== Best Model (LR + GridSearch) ===")
print(classification_report(y_test, y_pred_best))
plot_confusion_matrix(y_test, y_pred_best, 'Confusion Matrix — Best LR (GridSearch)', '07_cm_lr_best.png')

## Cell 8 — **comparison chart**

In [ ]:
# ── Model comparison chart ───────────────────────────────────────────────────
models = ['LR Baseline', 'SVM Baseline', 'kNN Baseline', 'LR Best (GridSearch)']
accuracy = [0.46, 0.49, 0.30, 0.55]
macro_f1 = [0.37, 0.36, 0.17, 0.40]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width/2, accuracy, width, label='Accuracy', color='steelblue')
ax.bar(x + width/2, macro_f1, width, label='Macro F1', color='coral')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('visuals/08_model_comparison.png')
plt.show()

# 05_imbalance_experiments.py

## Cell 1 — **imports, load data, train/test split, TF-IDF, helper function**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
import os

os.makedirs('visuals', exist_ok=True)

# ── Load and prepare data ────────────────────────────────────────────────────
df = pd.read_csv('data/processed/combined_dataset.csv')
X = df['text']
y = df['parrot_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

labels = sorted(y_test.unique())

# ── Helper: plot confusion matrix ────────────────────────────────────────────
def plot_confusion_matrix(y_test, y_pred, title, filename):
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'visuals/{filename}')
    plt.show()

## *COMBATTING FEAR* — **Minority Class Experiments**

### *Experiment 1: SMOTE* — 🔬

#### Cell 2 — **SMOTE setup + LR + SMOTE**

In [ ]:
print("=== Experiment 1: SMOTE ===")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_tfidf, y_train)
print(f"After SMOTE — training size: {len(y_train_smote)}")

lr_smote = LogisticRegression(random_state=42, max_iter=1000)
lr_smote.fit(X_train_smote, y_train_smote)
y_pred_lr_smote = lr_smote.predict(X_test_tfidf)
print("\n=== LR + SMOTE ===")
print(classification_report(y_test, y_pred_lr_smote))
plot_confusion_matrix(y_test, y_pred_lr_smote, 'Confusion Matrix — LR + SMOTE', '09_cm_lr_smote.png')


#### Cell 3 — **SVM + SMOTE**

In [ ]:
svm_smote = LinearSVC(random_state=42, max_iter=2000)
svm_smote.fit(X_train_smote, y_train_smote)
y_pred_svm_smote = svm_smote.predict(X_test_tfidf)
print("\n=== SVM + SMOTE ===")
print(classification_report(y_test, y_pred_svm_smote))
plot_confusion_matrix(y_test, y_pred_svm_smote, 'Confusion Matrix — SVM + SMOTE', '10_cm_svm_smote.png')

### *Experiment 2: LSA (Dimensionality Reduction)* — 📉

#### Cell 4 — **LSA setup + LR + LSA**

In [ ]:
print("\n=== Experiment 2: LSA ===")
svd = TruncatedSVD(n_components=300, random_state=42)
X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)
print(f"Reduced shape: {X_train_svd.shape}")

lr_svd = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr_svd.fit(X_train_svd, y_train)
y_pred_lr_svd = lr_svd.predict(X_test_svd)
print("\n=== LR + LSA ===")
print(classification_report(y_test, y_pred_lr_svd))
plot_confusion_matrix(y_test, y_pred_lr_svd, 'Confusion Matrix — LR + LSA', '11_cm_lr_lsa.png')

#### Cell 5 — **SVM + LSA**

In [ ]:
svm_svd = LinearSVC(random_state=42, max_iter=2000)
svm_svd.fit(X_train_svd, y_train)
y_pred_svm_svd = svm_svd.predict(X_test_svd)
print("\n=== SVM + LSA ===")
print(classification_report(y_test, y_pred_svm_svd, zero_division=0))
plot_confusion_matrix(y_test, y_pred_svm_svd, 'Confusion Matrix — SVM + LSA', '12_cm_svm_lsa.png')

### *Experiment 3: Class Weights* — ⚖️

#### Cell 6 — **Aggressive class weights + LR**

In [ ]:
print("\n=== Experiment 3: Aggressive Class Weights ===")
custom_weights = {
    'joy': 1, 'surprise': 2, 'anger': 2, 'love': 2,
    'neutral': 3, 'sadness': 5, 'fear': 20
}

lr_weighted = LogisticRegression(class_weight=custom_weights, random_state=42, max_iter=1000)
lr_weighted.fit(X_train_tfidf, y_train)
y_pred_lr_weighted = lr_weighted.predict(X_test_tfidf)
print("\n=== LR + Aggressive Weights ===")
print(classification_report(y_test, y_pred_lr_weighted))
plot_confusion_matrix(y_test, y_pred_lr_weighted, 'Confusion Matrix — LR + Aggressive Weights', '13_cm_lr_weighted.png')

#### Cell 7 — **Computed class weights + LR**

In [ ]:
print("\n=== Experiment 4: Computed Class Weights ===")
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
computed_weights = dict(zip(classes, weights))
print(f"Computed weights: {computed_weights}")

lr_computed = LogisticRegression(class_weight=computed_weights, random_state=42, max_iter=1000)
lr_computed.fit(X_train_tfidf, y_train)
y_pred_lr_computed = lr_computed.predict(X_test_tfidf)
print("\n=== LR + Computed Weights ===")
print(classification_report(y_test, y_pred_lr_computed))

## Cell 8 — **All experiments summary chart**

In [ ]:
# ── All experiments summary chart ────────────────────────────────────────────
experiments = [
    'LR Baseline', 'SVM Baseline', 'kNN Baseline',
    'LR + SMOTE', 'SVM + SMOTE',
    'LR + LSA', 'SVM + LSA',
    'LR + Aggressive Weights', 'LR + Computed Weights',
    'LR + GridSearch (Best)'
]

macro_f1_scores = [0.37, 0.36, 0.17, 0.37, 0.35, 0.30, 0.25, 0.39, 0.37, 0.40]

plt.figure(figsize=(14, 6))
colors = ['coral' if s == max(macro_f1_scores) else 'steelblue' for s in macro_f1_scores]
plt.bar(experiments, macro_f1_scores, color=colors)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Macro F1 Score')
plt.title('COMBATTING FEAR — All Experiments Macro F1 Comparison')
plt.ylim(0, 0.6)
plt.tight_layout()
plt.savefig('visuals/14_all_experiments_comparison.png')
plt.show()

# 06_genetic_algorithm.py

## Cell 1 — **imports + data setup**

In [ ]:
!pip install deap

In [ ]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from deap import base, creator, tools, algorithms
import os

os.makedirs('visuals', exist_ok=True)

# ── Load and prepare data ────────────────────────────────────────────────────
df = pd.read_csv('data/processed/combined_dataset.csv')
X = df['text']
y = df['parrot_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

labels = sorted(y_test.unique())
emotion_classes = ['anger', 'fear', 'joy', 'love', 'neutral', 'sadness', 'surprise']

## Cell 2 — fitness function + GA Configuration

In [ ]:
# ── Fitness function ─────────────────────────────────────────────────────────
def evaluate(individual):
    weights = {emotion_classes[i]: max(individual[i], 0.1) for i in range(7)}
    clf = LogisticRegression(C=0.5, class_weight=weights, random_state=42, max_iter=500)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in skf.split(X_train_tfidf, y_train):
        X_tr, X_val = X_train_tfidf[train_idx], X_train_tfidf[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_val)
        scores.append(f1_score(y_val, y_pred, average='macro'))
    return (np.mean(scores),)

# ── GA setup ─────────────────────────────────────────────────────────────────
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_float", random.uniform, 0.1, 25.0)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=7)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=2, indpb=0.3)
toolbox.register("select", tools.selTournament, tournsize=3)


## Cell 3 — run GA + print results

In [ ]:
# ── Run GA ───────────────────────────────────────────────────────────────────
random.seed(42)
np.random.seed(42)

population = toolbox.population(n=10)
NGEN = 5
best_per_gen = []

print("Running Genetic Algorithm...")
for gen in range(NGEN):
    offspring = algorithms.varAnd(population, toolbox, cxpb=0.5, mutpb=0.3)
    fits = list(map(toolbox.evaluate, offspring))
    for fit, ind in zip(fits, offspring):
        ind.fitness.values = fit
    population = toolbox.select(offspring, k=len(population))
    best = tools.selBest(population, k=1)[0]
    best_per_gen.append(best.fitness.values[0])
    print(f"Gen {gen+1} | Best Macro F1: {best.fitness.values[0]:.4f}")

# ── Best weights found ───────────────────────────────────────────────────────
best_individual = tools.selBest(population, k=1)[0]
best_weights = {emotion_classes[i]: round(max(best_individual[i], 0.1), 3) for i in range(7)}
print(f"\nBest weights found: {best_weights}")

# ── Train final model with GA weights ────────────────────────────────────────
lr_ga = LogisticRegression(C=0.5, class_weight=best_weights, random_state=42, max_iter=1000)
lr_ga.fit(X_train_tfidf, y_train)
y_pred_ga = lr_ga.predict(X_test_tfidf)

print("\n=== LR + Genetic Algorithm Weights ===")
print(classification_report(y_test, y_pred_ga))

## Cell 4 — confusion matrix + convergence plot

In [ ]:
# ── Confusion matrix ─────────────────────────────────────────────────────────
cm_ga = confusion_matrix(y_test, y_pred_ga, labels=labels)
plt.figure(figsize=(10, 7))
sns.heatmap(cm_ga, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.title('Confusion Matrix — LR + GA Weights')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('visuals/15_cm_lr_ga.png')
plt.show()

# ── GA convergence plot ───────────────────────────────────────────────────────
plt.figure(figsize=(8, 5))
plt.plot(range(1, NGEN+1), best_per_gen, marker='o', color='steelblue')
plt.title('GA Convergence — Best Macro F1 per Generation')
plt.xlabel('Generation')
plt.ylabel('Macro F1')
plt.tight_layout()
plt.savefig('visuals/16_ga_convergence.png')
plt.show()